# Computational Theory - Jamie Walsh

## Contents

- [Problem 1: Representing SHA-256 data](#problem-1-representing-sha-256-data)
- [Problem 2: SHA-256 Bitwise Operations](#problem-2-sha-256-bitwise-operations)

In [11]:
import numpy as np
import sys
import struct

## Problem 1: Representing SHA-256 data

Before beginning any work on implementing SHA-256, this section will cover how we represent SHA-256's inputs, outputs and intermediate data in Python, and why these representations are appropriate.

### 32-bit words

Why 32 bits? SHA-256 requires 32-bit words, as specified in [FIPS 180-4 Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13). Within that detail there's a constraint which is important for how we represent this data: values are limited to 32 bits and overflow if this limited is exceeded.

Unlike "lower-level" languages like C++, where a 32-bit integer comes as a built-in `uint32_t`, Python's `int` does not behave this way.

We need to use a data type that won't exceed 32 bits. To test this, we can take the max value and add 1. Python's plain `int` has no clean way to enforce this limit; however, [NumPy's `np.iinfo`](https://stackoverflow.com/questions/23189506/maximum-allowed-value-for-a-numpy-data-type) provides a solution.

In [12]:
limit = np.iinfo(np.uint32).max

print(f"int: {limit + 1}")  # plain int: no limit, keeps growing
print(f"np.uint32: {np.uint32(limit) + 1}")  # np.uint32: wraps back to 0

int: 4294967296
np.uint32: 0


/var/folders/3q/zzb1pzp9655bb6h57vz921yh0000gn/T/ipykernel_50047/1439755418.py:4: RuntimeWarning: overflow encountered in scalar add
  print(f"np.uint32: {np.uint32(limit) + 1}")  # np.uint32: wraps back to 0


This confirms we need `np.uint32` as it wraps the 32-bit limit while Python's plain `int` does not, which FIPS 180-4 requires.

### Sequences of 32-bit words
Also specified in [FIPS 180-4 Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13), SHA-256 represents each 512-bit message block as a sequence of sixteen 32-bit words. To represent this, we use a NumPy array over a standard list, as [NumPy's documentation](https://numpy.org/doc/stable/user/whatisnumpy.html) states arrays are more efficient when every element is the same type, which applies here as every element is a `uint32`.

We can verify this claim in code:


In [13]:
numpy_array = np.array([1, 2, 3, 4, 5], dtype=np.uint32)
python_list = [np.uint32(x) for x in [1, 2, 3, 4, 5]]

print(f"Numpy Array Size: {numpy_array.nbytes}")  # array's total memory

# lists total memory: size of pointers + size of every object they point to
print(f"Python List Size: {sys.getsizeof(python_list) + sum(sys.getsizeof(x) for x in python_list)}")

print("numpy array:")
%timeit numpy_array + 1

print("python list:")
%timeit [x + 1 for x in python_list]

Numpy Array Size: 20
Python List Size: 260
numpy array:
480 ns ± 6.06 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
python list:
247 ns ± 1.32 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


From this data, the array (20 bytes) is much more efficient than the list (260 bytes) which supports NumPy's claims. However, we can see that the array was actually slower here (472ns vs 238ns) so why?

NumPy has a small, fixed overhead every time it performs an operation. With only 5 elements, that overhead outweighs any benefit. NumPy's speed advantage only shows up once arrays are large enough to outweigh this fixed cost.

So for our case, is this overhead actually an issue? Later problems use larger sequences, for example the 64-word message schedule in Problem 5. So we retest below at the same size, rather than assume this result generalises.


In [14]:
numpy_array = np.array(range(0, 64), dtype=np.uint32)
python_list = [np.uint32(x) for x in range(0, 64)]

print("numpy array:")
%timeit numpy_array + 1

print("python list:")
%timeit [x + 1 for x in python_list]

numpy array:
483 ns ± 7.42 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
python list:
2.43 μs ± 61.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


As predicted, retesting at a larger size (64 elements) shows the array is now significantly faster (486ns vs 2.44μs), roughly 5x faster. This confirms the array remains the right choice.

### Input Messages

Input messages can be stored as strings and converted to bytes using `.encode()` when required.

In [15]:
message = "abc"
message_bytes = message.encode()
print(message_bytes)

b'abc'


This confirms `.encode()` produces `bytes`, ready for further processing.

### 512-bit Message Blocks

A 512-bit message block is sixteen words. A block starts out as 64 bytes, so representing it as 16 words requires expressing those bytes as words. At a low level, these are all just bits, so rather than converting them (which costs computation), we can simply change how they're represented.

In [16]:
example_list = list(range(1, 17))

packed_bytes = struct.pack('>16I', *example_list)
unpacked = struct.unpack('>16I', packed_bytes)  # struct.unpack: reads raw bytes back as a tuple of ints, no computation
print(f"Round-trip correct: {unpacked == tuple(example_list)}")

print(f"List size: {sys.getsizeof(example_list) + sum(sys.getsizeof(x) for x in example_list)}")
print(f"Packed bytes size: {sys.getsizeof(packed_bytes)}")

example_bytes = b'\x00\x00\x00\x05'

print("manual bit-shifting:")
%timeit (example_bytes[0] << 24) | (example_bytes[1] << 16) | (example_bytes[2] << 8) | example_bytes[3]

print("struct.unpack:")
%timeit struct.unpack('>I', example_bytes)[0]

words_array = np.frombuffer(packed_bytes, dtype='>u4')
print(f"Words array: {words_array}")

Round-trip correct: True
List size: 632
Packed bytes size: 97
manual bit-shifting:
73.9 ns ± 1.3 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
struct.unpack:
51.3 ns ± 0.467 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
Words array: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]


The results confirm two things. First, the round trip check shows the conversion is lossless. Unpacking the packed bytes returns exactly the original sixteen numbers, so no information is lost by reinterpreting rather than computing. Second, the size and speed results show that reinterpreting bytes as words (type punning) is both more compact and faster than either storing the values in a list or manually reconstructing each word with bit-shifting

When reading through the problems, I remembered seeing clever time and cost saving measures with bits and bytes from the [Fast Inverse Square Root algorithm](https://down2core.com/docs/lomont.pdf). It uses the same underlying trick: reading the same bits through a different type, with no computation involved. The difference is what it's used for. Fast Inverse Square Root trades accuracy for speed, using this trick to get an approximate answer. Here, the same trick is used the opposite way, for an exact, lossless reformatting of data, since SHA-256 needs every value to be bit-exact

Also of note: Two format strings are used here `>16I` and `>u4`. Both start with `>` meaning big-endian, the reason this is important will be touched on more below.

### The final 256-bit hash value

[FIPS 180-4 Section 2.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=9) states that `H_0^(i)` "is the left-most word of hash value i". Since the hash value is 256 bits and a word is 32 bits, the hash value is exactly 8 words, numbered 0 to 7, left to right:

[0] [1] [2] [3] [4] [5] [6] [7]

In [17]:
hash_words = np.array(range(1, 9), dtype='>u4')  # 8 words matching [0] ... [7]
hash_bytes =  hash_words.tobytes()  # words -> raw bytes, big-endian ('>u4') so word order matches [0] ... [7]
hex_string = hash_bytes.hex()  # bytes -> 64-character hex string, the final output format
print(hex_string)
print(len(hex_string))

0000000100000002000000030000000400000005000000060000000700000008
64


### Big-endian integers 

Back to the standard: [FIPS 180-4 Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=12) states that "Throughout this specification, the “big-endian” convention is used when expressing both 32- and 64-bit words, so that within each word, the most significant bit is stored in the left-most bit position".

So far we've been using `np.uint32` directly to represent a 32-bit word for arithmetic. Words, sequences, blocks and the hash value have all used it the same way without issue. This works everywhere except when converting to or from raw bytes. 

If we have a number `1234` and split it into `1`,`2`,`3`,`4`. We now have to decide which order to put them in, most significant first: (`1`,`2`,`3`,`4`) or least significant first: (`4`,`3`,`2`,`1`). The first is what "big-endian" means. The same applies to a `uint32`, converting it into 4 separate bytes splits it into pieces, and "which byte comes first" has to be consistent. A machine's native byte order may not answer the way FIPS 180-4 requires.

In [18]:
u32 = np.uint32  # native byte order -- safe for arithmetic
u32_be = '>u4'  # explicit big-endian -- required when converting to/from raw bytes 

Going forward we will use two short aliases to make this distinction more explicit: `u32` for ordinary arithmetic, and `u32_be` specifically for the moments a word needs to become, or come from, raw bytes, where big-endian must be stated rather than assumed.

### Summary of representation choices
- A **word** is `u32` (i.e. `np.uint32`), which enforces SHA-256's wraparound behavior
- A **sequence of words**, a **block** (16 words), and the **hash value** (8 words) are the same thing: a NumPy array of `u32` (i.e. `np.uint32`), with different lengths
- An **input message** is `bytes`, converted from a string with `.encode()`
- Bytes will be converted into words using **type punning** (`struct/np.frombuffer`) rather than manual computation
- **Big-endian** exists more as a rule for how conversions must be done rather than a type itself. `u32/u32_be` make this distinction explicit. 

## Problem 2: SHA-256 Bitwise Operations

SHA-256 will require some Boolean and bitwise functions for its actual implementation. These are seen in [FIPS 180-4 Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15) as follows:
1. `Ch(x, y, z)`
2. `Maj(x, y, z)`
3. `Sigma0(x)` — written as Σ0{256}(x) in the standard
4. `Sigma1(x)` — written as Σ1{256}(x) in the standard
5. `sigma0(x)` — written as σ0{256}(x) in the standard
6. `sigma1(x)` — written as σ1{256}(x) in the standard

This section will cover each one's implementation, along with a brief explanation and test for each.

Since several of these functions take the same 3 boolean inputs (`x, y, z`), a shared testing setup is defined once here and reused for each, rather than repeating it per function. `run_truth_table` is generic and works for any number of inputs; `xyz` specifically enumerates all 8 combinations for a 3-input boolean function, reused by both `Ch` and `Maj` since they share that signature. Only the expected outputs differ per function.

In [ ]:
xyz = [
    (0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1),
    (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1),
]


def run_truth_table(func, inputs, expected_values):
    for values, expected in zip(inputs, expected_values):
        result = func(*values)
        print(f"{func.__name__}{values} = {result}, expected {expected}: {result == expected}")

### Ch(x, y, z)

The Ch function: $Ch(x,y,z) = (x \land y) \oplus (\lnot x \land z)$

Broken down in pseudocode this becomes:
```pseudo
for each bit i:
    if x[i] == 1:
        result[i] = y[i]
    else:
        result[i] = z[i]
```

In [21]:
def Ch(x: u32, y: u32, z: u32) -> u32:
    """Ch function: For each bit positon, select"""
    return (x & y) ^ (~x & z)

In [ ]:
ch_expected = [0, 1, 0, 1, 0, 0, 1, 1]
run_truth_table(Ch, xyz, ch_expected)

### Maj(x, y, z)

The Maj function: $Maj(x,y,z) =  (x \land y) \oplus (x \land z) \oplus ( y \land z)$

Broken down in pseudocode this becomes:
```pseudo
for each bit i:
    count = x[i] + y[i] + z[i]
    if count is >= to 2:
        result[i] = 1
    else:
        result[i] = 0
```

In [ ]:
def Maj(x: u32, y: u32, z: u32) -> u32:
    """Maj function: For each bit positon, result is whichever value (0 or 1) appears in atleast two of x, y, z."""
    return (x & y) ^ (x & z) ^ (y & z)

In [ ]:
maj_expected = [0, 0, 0, 1, 0, 1, 1, 1]
run_truth_table(Maj, xyz, maj_expected)